In [1]:
# --- Colab bootstrap -------------------------------------------------------
# No-op when you already have the thermo package alongside this notebook (the
# normal case: you cloned the repository and are running from code/chNN/).
# In Colab there is no repository, so fetch the package and the property data.
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    # rm -rf first: git clone REFUSES an existing directory, and under `!` that
    # failure is silent -- a half-finished earlier clone would otherwise leave an
    # empty thermo/ and surface as a baffling ImportError further down.
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------

# Table 6.4-4 &mdash; thermodynamic properties of oxygen

This notebook **generates** Table 6.4-4: the compressibility factor $Z$, molar
volume $\underline{V}$, molar enthalpy $\underline{H}$, and molar entropy
$\underline{S}$ of oxygen from the Peng&ndash;Robinson equation of state, at
13 pressures and 11 temperatures. It is the table half of Illustration 6.4-1;
the three companion notebooks plot the same calculation as Figs. 6.4-3, 6.4-4
and 6.4-5.

Everything rests on one idea: a real-fluid
property is the **ideal-gas property plus a departure**, and the departure comes
entirely from the equation of state.

$$\underline{H}(T,P) = \underbrace{\int_{T_{\mathrm{ref}}}^{T} C_P^{*}\,dT'}_{\text{ideal gas}} + \underbrace{\bigl(\underline{H}-\underline{H}^{\mathrm{IG}}\bigr)_{T,P}}_{\text{from the EOS}}$$

### Inputs, and where each one comes from

| quantity | value | source |
|---|---|---|
| $T_c$ | 154.6 K | Table 6.6-1 |
| $P_c$ | 5.046 MPa | Table 6.6-1 |
| $\omega$ | 0.021 | Table 6.6-1 |
| $\kappa$ | computed from $\omega$, Eq. 6.7-4 | *not* entered as a rounded constant |
| $C_P^{*}$ | Appendix A.II, **cryogenic range** (100&ndash;700 K) | see below |
| reference state | ideal gas at 25&nbsp;$^{\circ}$C, 1 bar | so $\underline{H}=\underline{S}=0$ there |

Three deliberate choices about precision, because this notebook is the authority
for the printed table:

1. **$\kappa$ is computed, not entered.** Eq. 6.7-4 with $\omega=0.021$ gives
   $\kappa = 0.406908\ldots$, which is what the illustration prints as 0.4069.
   Entering the rounded 0.4069 instead would put a rounding error into every cell
   of the table.
2. **The gas constant is the SI value**, $R = 8.31446261815324$ J/(mol&nbsp;K),
   carried at full double precision throughout, as are the cubic's roots.
3. **The ideal-gas $C_P^{*}$ is the cryogenic-range row, not the 273&ndash;1800 K
   row.** This is the one that matters, by two orders of magnitude &mdash; see the
   next section.

The critical constants come from the book's own Table 6.6-1 rather than from
`pure_property.csv`, whose oxygen row carries $\omega = 0.025$ and
$P_c = 50.4$ bar &mdash; a different parameter set, which would put this table on a
different basis from the illustration that introduces it.

### Why the heat capacity dominates the accuracy of this table

The table runs from $-100\,^{\circ}$C = 173.15 K. Appendix A.II's familiar
*Low Temperature Range* row for oxygen is valid **273&ndash;1800 K**, so every cell
in the four coldest columns extrapolates it &mdash; and the extrapolation is not
benign:

| | $C_P^{*}$ at 173.15 K | error | costs, in the table |
|---|---|---|---|
| A.II 273&ndash;1800 K row, extrapolated | 27.883 | **&minus;1.235** (4.2%) | **67 J/mol** in $\underline{H}$, **0.32 J/(mol K)** in $\underline{S}$ |
| A.II cryogenic row (used here) | 29.069 | &minus;0.049 | 2.8 J/mol, 0.011 J/(mol K) |
| NIST-JANAF reference | 29.118 | &mdash; | &mdash; |

For scale: the gas constant contributes 0.02 J/mol and the published table's own
arithmetic error about 0.6 J/mol. **The heat capacity contributed a hundred times
either.** A.II's 273&ndash;1800 K row is not a poor fit &mdash; refitting that same
range reproduces its accuracy almost exactly &mdash; it is a fit to the wrong range.
The cryogenic row is the same cubic form fitted over 100&ndash;700 K, which for
oxygen is exactly the segment NIST-JANAF's own correlation covers.

Because $C_P^{*}$ enters only the ideal-gas term, this changes $\underline{H}$ and
$\underline{S}$ by one number per temperature column and leaves $Z$ and
$\underline{V}$ untouched.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst  
August 2026

In [2]:
import sys
sys.path.insert(0, "..")           # so `import thermo` finds code/thermo

import numpy as np
import pandas as pd

from thermo import PengRobinson, APPENDIX_A2_CP, APPENDIX_A2_CP_CRYO
from thermo.peng_robinson import R

# Oxygen, from the book's Table 6.6-1 and Appendix A.II. Built explicitly rather
# than with `from_database`: see the note above on the two differing omega values.
# The CRYOGENIC Cp* row, because this table starts at 173.15 K -- see thermo/data.py.
pr = PengRobinson(Tc=154.6, Pc=5.046e6, omega=0.021, name="oxygen",
                  cp=APPENDIX_A2_CP_CRYO["oxygen"])

cA, cB, cC, cD = pr.cp          # Cp* = cA + cB T + cC T^2 + cD T^3, J/(mol K)
T_REF, P_REF = 298.15, 1e5      # ideal gas at 25 C, 1 bar -> H = 0, S = 0
TO_KELVIN, BAR = 273.15, 1e5

# The table's axes.
T_C = [-100, -75, -50, -25, 0, 25, 50, 75, 100, 125, 150]
P_BAR = [1, 2, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

print(f"R     = {R!r} J/(mol K)")
print(f"Tc    = {pr.Tc} K      Pc = {pr.Pc/1e6:.3f} MPa")
print(f"omega = {pr.omega}   ->  kappa = {pr.kappa!r}")
print(f"                          prints as {pr.kappa:.4f}")
print(f"b     = {pr.b:.6e} m3/mol")

# What the choice of Cp* row costs, at the two ends of the table.
def _cp(c, T):
    return c[0] + c[1]*T + c[2]*T**2 + c[3]*T**3

print("\nCp* J/(mol K)      173.15 K   423.15 K")
for label, c in (("A.II 273-1800 K", APPENDIX_A2_CP["oxygen"]),
                 ("A.II cryogenic ", APPENDIX_A2_CP_CRYO["oxygen"])):
    print(f"  {label}  {_cp(c, 173.15):9.3f}  {_cp(c, 423.15):9.3f}")
print("  NIST-JANAF        29.118     30.314   <- the reference both are fitted to")

R     = 8.31446261815324 J/(mol K)
Tc    = 154.6 K      Pc = 5.046 MPa
omega = 0.021   ->  kappa = 0.40690842527999993
                          prints as 0.4069
b     = 1.981874e-05 m3/mol

Cp* J/(mol K)      173.15 K   423.15 K
  A.II 273-1800 K     27.883     30.707
  A.II cryogenic      29.068     30.328
  NIST-JANAF        29.118     30.314   <- the reference both are fitted to


## I. Volume

At every $(T,P)$ the cubic form of the EOS, Eq. 6.4-4,

$$Z^3 + (-1+B)Z^2 + (A-3B^2-2B)Z + (-AB+B^2+B^3) = 0,
\qquad A=\frac{a(T)P}{(RT)^2},\quad B=\frac{bP}{RT}$$

is solved for $Z$, and then $\underline{V} = ZRT/P$.

Every state in the table is **supercritical in temperature** &mdash; the coldest
column, $-100\,^{\circ}$C = 173.15 K, is already above $T_c = 154.6$ K &mdash; so
the cubic has one physically meaningful root everywhere and no vapor/liquid choice
arises. That is checked rather than assumed.

In [3]:
assert min(T_C) + TO_KELVIN > pr.Tc, "a column has dropped below Tc"

Z = np.empty((len(P_BAR), len(T_C)))
V = np.empty_like(Z)

for j, P in enumerate(P_BAR):
    for i, t in enumerate(T_C):
        T, Pa = t + TO_KELVIN, P * BAR
        roots = pr.physical_Z(T, Pa)
        assert len(roots) == 1, f"{len(roots)} physical roots at {t} C, {P} bar"
        Z[j, i] = roots[0]
        V[j, i] = roots[0] * R * T / Pa * 1e3      # m^3/kmol

print(f"Z from {Z.min():.4f} to {Z.max():.4f}")
print(f"one real root at all {Z.size} states, as expected above Tc")

Z from 0.4296 to 1.0010
one real root at all 143 states, as expected above Tc


## II. Enthalpy

Relative to the ideal gas at 25&nbsp;$^{\circ}$C and 1 bar, Eq. 6.4-19 with the
reference-state departure equal to zero by construction:

$$\underline{H}(T,P) = \int_{T_{\mathrm{ref}}}^{T} C_P^{*}(T')\,dT'
+ \bigl(\underline{H}-\underline{H}^{\mathrm{IG}}\bigr)_{T,P}$$

and the PR departure is Eq. 6.4-29,

$$\bigl(\underline{H}-\underline{H}^{\mathrm{IG}}\bigr)_{T,P}
= RT(Z-1) + \frac{T\,\dfrac{da}{dT} - a}{2\sqrt{2}\,b}
\ln\!\left[\frac{Z+(1+\sqrt2)B}{Z+(1-\sqrt2)B}\right]$$

Note what the reference state does **not** do: it does not make
$\underline{H}=0$ at 25&nbsp;$^{\circ}$C and 1 bar. The reference is the *ideal
gas* at those conditions, and real oxygen there is not quite ideal ($Z = 0.9991$),
so the table's own value at that state is the departure alone &mdash; a small
negative number rather than zero. It is the one cell that shows the reader what
the departure function is.

In [4]:
def H_ideal(T):
    """Ideal-gas enthalpy relative to T_REF, J/mol (Appendix A.II Cp)."""
    return (cA * (T - T_REF) + cB / 2 * (T**2 - T_REF**2)
            + cC / 3 * (T**3 - T_REF**3) + cD / 4 * (T**4 - T_REF**4))


H = np.empty_like(Z)
for j, P in enumerate(P_BAR):
    for i, t in enumerate(T_C):
        T, Pa = t + TO_KELVIN, P * BAR
        H[j, i] = H_ideal(T) + pr.departure_H(T, Pa)

i25, j1 = T_C.index(25), P_BAR.index(1)
print(f"H(25 C, 1 bar) = {H[j1, i25]:.2f} J/mol   "
      f"(the departure alone; Z = {Z[j1, i25]:.4f})")

H(25 C, 1 bar) = -9.45 J/mol   (the departure alone; Z = 0.9991)


## III. Entropy

The same decomposition, with the ideal-gas part now carrying the pressure term:

$$\underline{S}(T,P) = \int_{T_{\mathrm{ref}}}^{T}\frac{C_P^{*}(T')}{T'}\,dT'
- R\ln\frac{P}{P_{\mathrm{ref}}}
+ \bigl(\underline{S}-\underline{S}^{\mathrm{IG}}\bigr)_{T,P}$$

and the PR departure is Eq. 6.4-30,

$$\bigl(\underline{S}-\underline{S}^{\mathrm{IG}}\bigr)_{T,P}
= R\ln(Z-B) + \frac{\dfrac{da}{dT}}{2\sqrt{2}\,b}
\ln\!\left[\frac{Z+(1+\sqrt2)B}{Z+(1-\sqrt2)B}\right]$$

The $-R\ln(P/P_{\mathrm{ref}})$ term is why the $\underline{S}$ column falls so
much faster with pressure than $\underline{H}$ does: most of that variation is
ideal-gas behavior, not departure.

In [5]:
def S_ideal(T, P):
    """Ideal-gas entropy relative to (T_REF, P_REF), J/(mol K)."""
    return (cA * np.log(T / T_REF) + cB * (T - T_REF)
            + cC / 2 * (T**2 - T_REF**2) + cD / 3 * (T**3 - T_REF**3)
            - R * np.log(P / P_REF))


S = np.empty_like(Z)
for j, P in enumerate(P_BAR):
    for i, t in enumerate(T_C):
        T, Pa = t + TO_KELVIN, P * BAR
        S[j, i] = S_ideal(T, Pa) + pr.departure_S(T, Pa)

print(f"S(25 C, 1 bar) = {S[j1, i25]:.2f} J/(mol K)   (the departure alone)")

S(25 C, 1 bar) = -0.02 J/(mol K)   (the departure alone)


## Table 6.4-4

Assembled in the printed layout: one block per pressure, temperature across.

Column widths follow the printed table &mdash; $Z$ and $\underline{V}$ to four
decimals, $\underline{H}$ and $\underline{S}$ to two. Every digit shown is a
computed digit, which the published
$\underline{V}$ row's were not: its values were rounded to four *significant* figures
and then padded out to four decimals, so a cell printed as `14.3200` was really
`14.3163`.

The two decimals on $\underline{H}$ and $\underline{S}$ are *arithmetic* precision,
not accuracy. The Peng&ndash;Robinson equation and the extrapolated $C_P^{*}$ are
both far coarser than 0.01 J/mol; the digits are carried so that a reader who reruns
this notebook can confirm cell by cell that they have reproduced it.

In [6]:
UNITS = ("V [=] m3/kmol;  H [=] J/mol = kJ/kmol;  "
         "S [=] J/(mol K) = kJ/(kmol K)")

# Fixed decimals per column, matching the printed table's own widths, so that a
# reader comparing this output with the book is comparing like with like. Trailing
# zeros are kept: a padded zero here is a real digit, which in the published table it
# was not (its V column was rounded to four significant figures and then padded
# out to four decimals, so "14.3200" claimed two digits it did not have).
DECIMALS = {"Z": 4, "V": 4, "H": 2, "S": 2}


def fmt(kind, x):
    return f"{x:.{DECIMALS[kind]}f}"


W = 11
lines = ["Table 6.4-4  Thermodynamic Properties of Oxygen Calculated Using the "
         "Peng-Robinson Equation of State", ""]
for j, P in enumerate(P_BAR):
    lines.append(f"P = {P} bar")
    lines.append("T (C)".ljust(8) + "".join(f"{t:>{W}}" for t in T_C))
    for kind, grid in (("Z", Z), ("V", V), ("H", H), ("S", S)):
        lines.append(kind.ljust(8)
                     + "".join(f"{fmt(kind, v):>{W}}" for v in grid[j]))
    lines.append("")
lines.append(UNITS)

table = "\n".join(lines)
print(table)

Table 6.4-4  Thermodynamic Properties of Oxygen Calculated Using the Peng-Robinson Equation of State

P = 1 bar
T (C)          -100        -75        -50        -25          0         25         50         75        100        125        150
Z            0.9945     0.9963     0.9974     0.9982     0.9987     0.9991     0.9993     0.9996     0.9997     0.9998     0.9999
V           14.3170    16.4135    18.5056    20.5946    22.6814    24.7667    26.8507    28.9339    31.0163    33.0982    35.1796
H          -3668.51   -2937.83   -2207.63   -1476.77    -744.32      -9.45     728.56    1470.32    2216.41    2967.29    3723.39
S            -15.93     -11.99      -8.51      -5.41      -2.60      -0.02       2.35       4.56       6.63       8.58      10.42

P = 2 bar
T (C)          -100        -75        -50        -25          0         25         50         75        100        125        150
Z            0.9889     0.9925     0.9948     0.9963     0.9974     0.9982     0.9987     0.9991 

### Files written

`output/Table_6.4-4.txt` is the table in the printed layout; `output/Table_6.4-4.csv` is the same numbers in tidy form (one row per state), which is the useful shape for typesetting and for checking a single cell.

In [7]:
os.makedirs("output", exist_ok=True)

with open("output/Table_6.4-4.txt", "w") as fh:
    fh.write(table + "\n")

tidy = pd.DataFrame(
    [{"P_bar": P, "T_C": t, "Z": Z[j, i], "V_m3_per_kmol": V[j, i],
      "H_J_per_mol": H[j, i], "S_J_per_mol_K": S[j, i]}
     for j, P in enumerate(P_BAR) for i, t in enumerate(T_C)])
tidy.to_csv("output/Table_6.4-4.csv", index=False)

print(f"{len(tidy)} states x 4 properties = {4*len(tidy)} values")
tidy.head()

143 states x 4 properties = 572 values


,P_bar,T_C,Z,V_m3_per_kmol,H_J_per_mol,S_J_per_mol_K
0,1,-100,0.994482,14.317050,-3668.505102,-15.926994
1,1,-75,0.996263,16.413547,-2937.834017,-11.985218
2,1,-50,0.997405,18.505573,-2207.630502,-8.514716
3,1,-25,0.998169,20.594560,-1476.772954,-5.410392
4,1,0,0.998698,22.681393,-744.317272,-2.598185


### What this table is, and is not

It is the Peng&ndash;Robinson equation evaluated exactly, on the book's own critical
constants, with an ideal-gas heat capacity accurate over the range tabulated. It is
**not** a table of the best known properties of oxygen: for that, oxygen has a
fundamental equation of state explicit in the Helmholtz energy (Schmidt and Wagner,
1985), which is the source of the NIST values, and it is more accurate than any cubic
can be.

**Where the remaining error lives.** With the cryogenic $C_P^{*}$ row, the ideal-gas
part contributes at most 2.8 J/mol and 0.011 J/(mol&nbsp;K) over this range. Everything
larger than that is now the **Peng&ndash;Robinson equation itself** &mdash; the
departure functions &mdash; which is the honest state for a table whose caption says
*calculated using the Peng-Robinson equation of state*. The arithmetic is no longer
the limit, and neither is the heat capacity.

Comparing against NIST is a separate exercise, and it carries a trap: the NIST
tables use a different reference state, so only **differences between states** may be
compared, never absolute $\underline{H}$ and $\underline{S}$.

In [8]:
print(cA, cB, cC, cD)

30.171 -0.01293 4.236e-05 -2.5828e-08
